# PyTorch GPU Readiness Mini-Lab

Target runtime: Google Colab free GPU, ideally T4 when allocated.

This notebook trains a small tabular MLP on OpenML GiveMeSomeCredit, compares CPU vs GPU, enables mixed precision with `torch.amp`, profiles the GPU AMP run with PyTorch Profiler, and writes a JSON evidence artifact.

Before running: `Runtime > Change runtime type > GPU`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
import tracemalloc
import urllib.request
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.io import arff
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
OPENML_ARFF_URL = "https://openml.org/data/v1/download/22125240/GiveMeSomeCredit.arff"
OPENML_ID = 46929
DATASET_NAME = "GiveMeSomeCredit"
TARGET_COLUMN = "FinancialDistressNextTwoYears"
FEATURE_COLUMNS = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]

EPOCHS = 8
BATCH_SIZE = 4096
LEARNING_RATE = 3e-3
TRAIN_REPEAT_FACTOR = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_float32_matmul_precision("high")

cwd = Path.cwd()
if cwd.name == "pytorch-gpu-readiness-lab":
    LAB_ROOT = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "pytorch-gpu-readiness-lab":
    LAB_ROOT = cwd.parent
else:
    LAB_ROOT = cwd / "pytorch-gpu-readiness-lab"
DATA_DIR = LAB_ROOT / "data"
ARTIFACT_DIR = LAB_ROOT / "artifacts"
PROFILER_DIR = ARTIFACT_DIR / "profiler"
REPORT_DIR = LAB_ROOT / "reports"
for path in (DATA_DIR, ARTIFACT_DIR, PROFILER_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

RAW_ARFF_PATH = DATA_DIR / "GiveMeSomeCredit_openml_46929.arff"
METRICS_JSON_PATH = ARTIFACT_DIR / "gpu_readiness_metrics.json"
COLAB_REPORT_PATH = REPORT_DIR / "gpu_readiness_report_colab.md"
PROFILER_SUMMARY_PATH = PROFILER_DIR / "gpu_amp_profiler_summary.txt"

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def nvidia_smi() -> str | None:
    if not shutil.which("nvidia-smi"):
        return None
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip() or result.stderr.strip() or None

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "nvidia_smi": nvidia_smi(),
    "lab_root": str(LAB_ROOT),
})

## Data

The notebook uses the public OpenML curation of GiveMeSomeCredit (`data_id=46929`). The original Kaggle competition data is not fetched from Kaggle, so no Kaggle credentials are required.

In [ ]:
def download_dataset() -> Path:
    if RAW_ARFF_PATH.exists() and RAW_ARFF_PATH.stat().st_size > 0:
        return RAW_ARFF_PATH
    print(f"Downloading {DATASET_NAME} from OpenML to {RAW_ARFF_PATH}...")
    urllib.request.urlretrieve(OPENML_ARFF_URL, RAW_ARFF_PATH)
    return RAW_ARFF_PATH

def decode_if_bytes(value):
    return value.decode("utf-8") if isinstance(value, bytes) else value

def load_credit_dataframe(path: Path) -> tuple[pd.DataFrame, pd.Series]:
    records, _ = arff.loadarff(path)
    frame = pd.DataFrame(records)
    for column in frame.columns:
        if frame[column].dtype == object:
            frame[column] = frame[column].map(decode_if_bytes)
    y = frame[TARGET_COLUMN].map(lambda value: 1 if value in {"Yes", 1, "1", True} else 0).astype(np.float32)
    x = frame[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    return x, y

raw_path = download_dataset()
features_df, target_s = load_credit_dataframe(raw_path)

x_train_df, x_tmp_df, y_train_s, y_tmp_s = train_test_split(
    features_df,
    target_s,
    test_size=0.30,
    random_state=SEED,
    stratify=target_s,
)
x_val_df, x_test_df, y_val_s, y_test_s = train_test_split(
    x_tmp_df,
    y_tmp_s,
    test_size=0.50,
    random_state=SEED,
    stratify=y_tmp_s,
)

train_medians = x_train_df.median(numeric_only=True)
x_train_df = x_train_df.fillna(train_medians)
x_val_df = x_val_df.fillna(train_medians)
x_test_df = x_test_df.fillna(train_medians)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_df).astype(np.float32)
x_val = scaler.transform(x_val_df).astype(np.float32)
x_test = scaler.transform(x_test_df).astype(np.float32)
y_train = y_train_s.to_numpy(dtype=np.float32)
y_val = y_val_s.to_numpy(dtype=np.float32)
y_test = y_test_s.to_numpy(dtype=np.float32)

print({
    "rows": int(len(features_df)),
    "features": int(features_df.shape[1]),
    "positive_rate": float(target_s.mean()),
    "train_rows": int(len(x_train)),
    "validation_rows": int(len(x_val)),
    "test_rows": int(len(x_test)),
})

## Model and Benchmark Helpers

The GPU AMP run uses `torch.amp.autocast` and `torch.amp.GradScaler`. CPU, GPU FP32, and GPU AMP are measured with the same model shape, same split, same batch size, and same epoch count.

In [ ]:
x_train_t = torch.from_numpy(x_train)
y_train_t = torch.from_numpy(y_train).view(-1, 1)
x_val_t = torch.from_numpy(x_val)
y_val_t = torch.from_numpy(y_val).view(-1, 1)
x_test_t = torch.from_numpy(x_test)
y_test_t = torch.from_numpy(y_test).view(-1, 1)

if TRAIN_REPEAT_FACTOR > 1:
    x_train_bench_t = x_train_t.repeat((TRAIN_REPEAT_FACTOR, 1))
    y_train_bench_t = y_train_t.repeat((TRAIN_REPEAT_FACTOR, 1))
else:
    x_train_bench_t = x_train_t
    y_train_bench_t = y_train_t

class CreditMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def make_loader(x: torch.Tensor, y: torch.Tensor, device: torch.device, shuffle: bool) -> DataLoader:
    return DataLoader(
        TensorDataset(x, y),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

def process_rss_mb() -> float | None:
    try:
        import psutil

        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    except Exception:
        return None

def make_grad_scaler(enabled: bool):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=True)

def synchronize_if_needed(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)

@torch.no_grad()
def evaluate_model(model: nn.Module, device: torch.device) -> dict[str, float]:
    model.eval()
    loader = make_loader(x_test_t, y_test_t, device, shuffle=False)
    logits_out = []
    labels_out = []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb).detach().float().cpu().numpy().ravel()
        logits_out.append(logits)
        labels_out.append(yb.numpy().ravel())
    logits_np = np.concatenate(logits_out)
    labels_np = np.concatenate(labels_out)
    probs_np = 1.0 / (1.0 + np.exp(-logits_np))
    return {
        "roc_auc": float(roc_auc_score(labels_np, probs_np)),
        "average_precision": float(average_precision_score(labels_np, probs_np)),
    }

def train_benchmark(label: str, device: torch.device, use_amp: bool, profile: bool = False) -> dict[str, object]:
    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

    model = CreditMLP(input_dim=x_train_t.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    loader = make_loader(x_train_bench_t, y_train_bench_t, device, shuffle=True)
    amp_enabled = bool(use_amp and device.type == "cuda")
    scaler_amp = make_grad_scaler(amp_enabled)
    autocast_ctx = lambda: torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled)

    profiler = None
    profiler_context = nullcontext()
    if profile:
        activities = [torch.profiler.ProfilerActivity.CPU]
        if device.type == "cuda":
            activities.append(torch.profiler.ProfilerActivity.CUDA)
        profiler = torch.profiler.profile(
            activities=activities,
            schedule=torch.profiler.schedule(wait=1, warmup=1, active=4, repeat=1),
            on_trace_ready=torch.profiler.tensorboard_trace_handler(str(PROFILER_DIR)),
            record_shapes=True,
            profile_memory=True,
            with_stack=False,
        )
        profiler_context = profiler

    rss_before = process_rss_mb()
    tracemalloc.start()
    synchronize_if_needed(device)
    started = time.perf_counter()
    last_loss = math.nan
    samples_seen = 0

    with profiler_context:
        for _epoch in range(EPOCHS):
            model.train()
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with autocast_ctx() if amp_enabled else nullcontext():
                    logits = model(xb)
                    loss = loss_fn(logits, yb)
                if scaler_amp is not None:
                    scaler_amp.scale(loss).backward()
                    scaler_amp.step(optimizer)
                    scaler_amp.update()
                else:
                    loss.backward()
                    optimizer.step()
                samples_seen += int(xb.shape[0])
                last_loss = float(loss.detach().cpu())
                if profiler is not None:
                    profiler.step()

    synchronize_if_needed(device)
    wall_time_seconds = time.perf_counter() - started
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after = process_rss_mb()
    metrics = evaluate_model(model, device)

    profiler_summary = None
    if profiler is not None:
        sort_key = "cuda_time_total" if device.type == "cuda" else "cpu_time_total"
        profiler_summary = profiler.key_averages().table(sort_by=sort_key, row_limit=25)
        PROFILER_SUMMARY_PATH.write_text(profiler_summary, encoding="utf-8")

    payload = {
        "label": label,
        "status": "completed",
        "device": str(device),
        "precision": "torch.amp.float16" if amp_enabled else "fp32",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "train_repeat_factor": TRAIN_REPEAT_FACTOR,
        "samples_seen": samples_seen,
        "wall_time_seconds": round(wall_time_seconds, 6),
        "throughput_samples_per_second": round(samples_seen / wall_time_seconds, 3),
        "last_train_loss": round(last_loss, 6),
        "test_roc_auc": round(metrics["roc_auc"], 6),
        "test_average_precision": round(metrics["average_precision"], 6),
        "python_tracemalloc_peak_mb": round(peak_bytes / (1024 ** 2), 3),
        "process_rss_delta_mb": round((rss_after - rss_before), 3) if rss_before is not None and rss_after is not None else None,
    }
    if device.type == "cuda":
        payload.update({
            "cuda_device_name": torch.cuda.get_device_name(device),
            "gpu_peak_memory_allocated_mb": round(torch.cuda.max_memory_allocated(device) / (1024 ** 2), 3),
            "gpu_peak_memory_reserved_mb": round(torch.cuda.max_memory_reserved(device) / (1024 ** 2), 3),
        })
    if profiler_summary is not None:
        payload["profiler_summary_path"] = str(PROFILER_SUMMARY_PATH)
        payload["profiler_trace_dir"] = str(PROFILER_DIR)
    return payload

## Run CPU, GPU FP32, and GPU AMP

If Colab did not allocate a CUDA device, the notebook still writes a CPU baseline and a skipped-GPU artifact. Re-run after selecting a GPU runtime to get the readiness evidence.

In [ ]:
runs: list[dict[str, object]] = []

cpu_result = train_benchmark("cpu_fp32", torch.device("cpu"), use_amp=False, profile=False)
runs.append(cpu_result)
print("CPU baseline", cpu_result)

if torch.cuda.is_available():
    cuda_device = torch.device("cuda")
    gpu_fp32_result = train_benchmark("gpu_fp32", cuda_device, use_amp=False, profile=False)
    runs.append(gpu_fp32_result)
    print("GPU FP32", gpu_fp32_result)

    gpu_amp_result = train_benchmark("gpu_amp", cuda_device, use_amp=True, profile=True)
    runs.append(gpu_amp_result)
    print("GPU AMP", gpu_amp_result)
else:
    skipped = {
        "label": "gpu_amp",
        "status": "skipped_no_cuda",
        "message": "Select Runtime > Change runtime type > GPU in Colab and rerun all cells.",
    }
    runs.append(skipped)
    print(skipped)

## Save Artifact JSON and Report

The JSON is the main evidence artifact. It records actual hardware, throughput, memory usage, speedup, profiler paths, and metric sanity checks.

In [ ]:
def completed(label: str) -> dict[str, object] | None:
    for run in runs:
        if run.get("label") == label and run.get("status") == "completed":
            return run
    return None

def safe_ratio(numerator: float | None, denominator: float | None) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    return round(float(numerator) / float(denominator), 4)

cpu = completed("cpu_fp32")
gpu_fp32 = completed("gpu_fp32")
gpu_amp = completed("gpu_amp")

comparison = {
    "gpu_amp_vs_cpu_speedup_by_wall_time": safe_ratio(
        cpu.get("wall_time_seconds") if cpu else None,
        gpu_amp.get("wall_time_seconds") if gpu_amp else None,
    ),
    "gpu_amp_vs_cpu_speedup_by_throughput": safe_ratio(
        gpu_amp.get("throughput_samples_per_second") if gpu_amp else None,
        cpu.get("throughput_samples_per_second") if cpu else None,
    ),
    "gpu_amp_vs_gpu_fp32_speedup_by_throughput": safe_ratio(
        gpu_amp.get("throughput_samples_per_second") if gpu_amp else None,
        gpu_fp32.get("throughput_samples_per_second") if gpu_fp32 else None,
    ),
}

artifact = {
    "artifact_version": 1,
    "status": "completed" if gpu_amp else "cpu_only_no_cuda",
    "generated_at_utc": utc_now_iso(),
    "environment": {
        "runtime": "Google Colab" if "google.colab" in sys.modules else "local_or_unknown",
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
        "nvidia_smi": nvidia_smi(),
    },
    "dataset": {
        "name": DATASET_NAME,
        "openml_id": OPENML_ID,
        "source_url": OPENML_ARFF_URL,
        "rows": int(len(features_df)),
        "features": int(features_df.shape[1]),
        "target": TARGET_COLUMN,
        "positive_rate": round(float(target_s.mean()), 6),
        "train_rows": int(len(x_train)),
        "validation_rows": int(len(x_val)),
        "test_rows": int(len(x_test)),
    },
    "config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "train_repeat_factor": TRAIN_REPEAT_FACTOR,
        "optimizer": "AdamW",
        "loss": "BCEWithLogitsLoss",
        "mixed_precision_api": "torch.amp.autocast + torch.amp.GradScaler",
    },
    "runs": runs,
    "comparison": comparison,
    "outputs": {
        "metrics_json": str(METRICS_JSON_PATH),
        "profiler_summary": str(PROFILER_SUMMARY_PATH) if PROFILER_SUMMARY_PATH.exists() else None,
        "profiler_dir": str(PROFILER_DIR),
        "colab_report": str(COLAB_REPORT_PATH),
    },
    "evidence_boundary": "Colab GPU evidence only after execution. No CINECA, IT4LIA, or production lending claim.",
}

METRICS_JSON_PATH.write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")

def fmt(value) -> str:
    return "n/a" if value is None else str(value)

report_md = f'''# PyTorch GPU Readiness Colab Run Report

Generated UTC: `{artifact['generated_at_utc']}`

## Runtime

- Torch: `{artifact['environment']['torch']}`
- CUDA available: `{artifact['environment']['cuda_available']}`
- CUDA device: `{artifact['environment']['cuda_device_name']}`
- nvidia-smi: `{artifact['environment']['nvidia_smi']}`

## Dataset

- Name: `{DATASET_NAME}`
- OpenML id: `{OPENML_ID}`
- Rows: `{artifact['dataset']['rows']}`
- Features: `{artifact['dataset']['features']}`
- Target positive rate: `{artifact['dataset']['positive_rate']}`

## Benchmark

- CPU baseline throughput: `{fmt(cpu.get('throughput_samples_per_second') if cpu else None)}` samples/sec
- GPU AMP throughput: `{fmt(gpu_amp.get('throughput_samples_per_second') if gpu_amp else None)}` samples/sec
- GPU AMP vs CPU speedup by wall time: `{fmt(comparison['gpu_amp_vs_cpu_speedup_by_wall_time'])}`
- GPU AMP vs CPU speedup by throughput: `{fmt(comparison['gpu_amp_vs_cpu_speedup_by_throughput'])}`
- GPU AMP peak allocated memory MB: `{fmt(gpu_amp.get('gpu_peak_memory_allocated_mb') if gpu_amp else None)}`
- GPU AMP peak reserved memory MB: `{fmt(gpu_amp.get('gpu_peak_memory_reserved_mb') if gpu_amp else None)}`

## Outputs

- Metrics JSON: `{METRICS_JSON_PATH}`
- Profiler summary: `{PROFILER_SUMMARY_PATH if PROFILER_SUMMARY_PATH.exists() else 'not generated'}`
- Profiler trace dir: `{PROFILER_DIR}`

## Boundary

This is a Colab GPU readiness benchmark, not a production credit model and not a claim of CINECA or IT4LIA execution.
'''

COLAB_REPORT_PATH.write_text(report_md, encoding="utf-8")
print(json.dumps({
    "metrics_json": str(METRICS_JSON_PATH),
    "report": str(COLAB_REPORT_PATH),
    "profiler_summary": str(PROFILER_SUMMARY_PATH) if PROFILER_SUMMARY_PATH.exists() else None,
    "comparison": comparison,
}, indent=2))

## Interpretation Notes

Tabular MLPs can be too small to saturate a T4. Treat the speedup as a measured result, not a guaranteed outcome. The readiness signal is the complete GPU workflow: CUDA runtime check, `torch.amp`, profiler trace, memory accounting, and a reproducible JSON artifact.